In [1]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from keras import layers
from sklearn.model_selection import train_test_split

# =============================================================================
# CONFIGURATION: Update ROOT to match your local project directory.
# Expected structure (generated by data_mask_for_unet.ipynb):
#   ROOT/
#     CData-re/brain_window/     <- Hemorrhage CT images
#     MasksData-re/qu-masks/     <- Quad-level masks for hemorrhage images
#     Normal3000-re/images/      <- Normal CT images
#     Normal3000-re/qu-masks/    <- All-black masks for normal images
# =============================================================================
ROOT        = Path(r'path/to/your/project')  # Update this to your local path
IMG_DIR     = ROOT / 'CData-re' / 'brain_window'
MASK_DIR    = ROOT / 'MasksData-re' / 'qu-masks'
NORMAL_IMG  = ROOT / 'Normal3000-re' / 'images'
NORMAL_MASK = ROOT / 'Normal3000-re' / 'qu-masks'

# Training hyperparameters
Patch_Size   = 128   # Input image size (128x128 pixels)
Batch_size   = 4     # Number of images per training batch
Buffer_size  = 500   # Shuffle buffer size for training dataset
EPOCHS       = 20    # Maximum number of training epochs
Output_Class = 4     # Number of segmentation classes: background, brain tissue, skull, hemorrhage

# =============================================================================
# DATA LOADING
# Collect all hemorrhage and normal image/mask pairs, keeping only images
# for which a corresponding mask file exists.
# =============================================================================
hemo_imgs  = sorted([f for f in IMG_DIR.iterdir()     if f.suffix == '.jpg'])
hemo_masks = sorted([f for f in MASK_DIR.iterdir()    if f.suffix == '.jpg'])
norm_imgs  = sorted([f for f in NORMAL_IMG.iterdir()  if f.suffix == '.jpg'])
norm_masks = sorted([f for f in NORMAL_MASK.iterdir() if f.suffix == '.jpg'])

# Only retain images that have a matching mask file (by filename)
hemo_name = {f.name for f in hemo_imgs} & {f.name for f in hemo_masks}
norm_name = {f.name for f in norm_imgs} & {f.name for f in norm_masks}

all_imgs  = [str(IMG_DIR    / n) for n in sorted(hemo_name)] +             [str(NORMAL_IMG / n) for n in sorted(norm_name)]
all_masks = [str(MASK_DIR   / n) for n in sorted(hemo_name)] +             [str(NORMAL_MASK / n) for n in sorted(norm_name)]

print(f'  Hemorrhage pairs : {len(hemo_name)}')
print(f'  Normal pairs     : {len(norm_name)}')
print(f'  Total pairs      : {len(all_imgs)}')

# Split into train (80%), validation (10%), and test (10%) sets
imgs_tv, imgs_test, masks_tv, masks_test = train_test_split(
    all_imgs, all_masks, test_size=0.10, random_state=42)
imgs_train, imgs_val, masks_train, masks_val = train_test_split(
    imgs_tv, masks_tv, test_size=0.10/0.90, random_state=42)
print(f'Train: {len(imgs_train)}  Val: {len(imgs_val)}  Test: {len(imgs_test)}')

# =============================================================================
# DATA PIPELINE
# =============================================================================

def read_pair(img_path, mask_path):
    """
    Load and preprocess a single image/mask pair.
    - Images are resized to Patch_Size x Patch_Size and normalized to [0, 1]
    - Masks use 4 intensity levels [0, 85, 170, 255] which are mapped to
      integer class labels [0, 1, 2, 3] for use with sparse categorical loss
    """
    img  = tf.io.decode_jpeg(tf.io.read_file(img_path), channels=3)
    img  = tf.image.resize(img, [Patch_Size, Patch_Size])
    img  = tf.cast(img, tf.float32) / 255.0

    # Map mask pixel values [0, 85, 170, 255] to class indices [0, 1, 2, 3]
    mask = tf.io.decode_jpeg(tf.io.read_file(mask_path), channels=1)
    mask = tf.image.resize(mask, [Patch_Size, Patch_Size], method='nearest')
    mask = tf.cast(mask, tf.float32) * 3.0 / 255.0
    mask = tf.cast(mask, tf.int32)
    return img, mask

class Augment(keras.layers.Layer):
    """
    Custom augmentation layer that applies random flips and small rotations
    to both the image and its mask simultaneously, using the same seed to
    ensure the transformations stay synchronized between image and mask.
    Data augmentation reduces overfitting by exposing the model to more
    varied versions of the training images.
    """
    def __init__(self, seed=42):
        super().__init__()
        self.flip_img  = layers.RandomFlip(seed=seed)
        self.flip_mask = layers.RandomFlip(seed=seed)
        self.rot_img   = layers.RandomRotation(0.02, seed=seed)
        self.rot_mask  = layers.RandomRotation(0.02, seed=seed)

    def call(self, img, mask):
        img  = self.rot_img(self.flip_img(img))
        mask = self.rot_mask(self.flip_mask(mask))
        return img, mask

def make_dataset(imgs, masks, shuffle=False, augment=False):
    """
    Build a tf.data.Dataset pipeline from lists of image and mask paths.
    Augmentation is only applied to the training set, not validation or test.
    Prefetching overlaps data loading with model training for efficiency.
    """
    ds = tf.data.Dataset.from_tensor_slices((imgs, masks))
    ds = ds.map(read_pair, num_parallel_calls=tf.data.AUTOTUNE)
    if shuffle:
        ds = ds.shuffle(Buffer_size)
    ds = ds.batch(Batch_size)
    if augment:
        ds = ds.map(Augment(), num_parallel_calls=tf.data.AUTOTUNE)
    return ds.prefetch(tf.data.AUTOTUNE)

ds_train = make_dataset(imgs_train, masks_train, shuffle=True, augment=True)
ds_val   = make_dataset(imgs_val,   masks_val)
ds_test  = make_dataset(imgs_test,  masks_test)

# =============================================================================
# U-NET ARCHITECTURE
#
# U-Net uses an encoder-decoder structure with skip connections:
#
# - Encoder: A series of convolutional blocks that progressively downsample
#   the image, extracting increasingly abstract features at each level.
#   At each level, a copy of the feature map is saved (the "skip connection").
#
# - Bottleneck: The deepest layer, where the most abstract representation
#   of the image is captured before upsampling begins.
#
# - Decoder: Mirrors the encoder, progressively upsampling back to the
#   original resolution. At each level, the skip connection from the
#   corresponding encoder level is concatenated in, restoring spatial
#   detail that was lost during downsampling. This is what allows the
#   U-Net to make precise pixel-level predictions rather than just
#   rough approximations.
#
# - Output: A final 1x1 convolution produces one logit per class per pixel.
#   The argmax across classes gives the predicted segmentation label.
# =============================================================================

def build_unet(input_shape=(Patch_Size, Patch_Size, 3), num_classes=Output_Class):
    """Build and return the U-Net model."""
    inputs = inputs = layers.Input(shape=input_shape)
    x = layers.BatchNormalization()(inputs)
    x = layers.Activation('relu')(x)

    # --- Encoder ---
    # Each block applies two convolutions, saves the result as a skip connection,
    # then downsamples via max pooling (halving spatial dimensions each time).
    skips, filters = [], [64, 128, 256, 512]
    for f in filters:
        x = layers.Conv2D(f, 3, padding='same')(x)
        x = layers.BatchNormalization()(x)
        x = layers.Activation('relu')(x)
        x = layers.Conv2D(f, 3, padding='same')(x)
        x = layers.BatchNormalization()(x)
        x = layers.Activation('relu')(x)
        skips.append(x)           # Save feature map for skip connection
        x = layers.MaxPooling2D()(x)  # Downsample

    # --- Bottleneck ---
    # Deepest point of the network — captures the most abstract representation
    x = layers.Conv2D(1024, 3, padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Conv2D(1024, 3, padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)

    # --- Decoder ---
    # Each block upsamples via transposed convolution, then concatenates the
    # corresponding skip connection to restore spatial detail, then applies
    # two convolutions to refine the features.
    for f, skip in zip(reversed(filters), reversed(skips)):
        x = layers.Conv2DTranspose(f, 3, strides=2, padding='same')(x)  # Upsample
        x = layers.BatchNormalization()(x)
        x = layers.Activation('relu')(x)
        x = layers.concatenate([x, skip])  # Merge with encoder skip connection
        x = layers.Conv2D(f, 3, padding='same')(x)
        x = layers.BatchNormalization()(x)
        x = layers.Activation('relu')(x)
        x = layers.Conv2D(f, 3, padding='same')(x)
        x = layers.BatchNormalization()(x)
        x = layers.Activation('relu')(x)

    # Final convolution layers before output
    x = layers.Conv2D(64, 3, padding='same')(x)
    x = layers.Conv2D(64, 3, padding='same')(x)
    # Output layer: one logit per class per pixel (no softmax — handled by loss function)
    x = layers.Conv2D(num_classes, 3, padding='same')(x)
    return keras.Model(inputs, x)

keras.backend.clear_session()
model = build_unet()
model.summary()

# =============================================================================
# CUSTOM METRIC: Mean IoU
# Intersection over Union (IoU) measures how well predicted regions overlap
# with true regions. Mean IoU averages this across all classes.
# We override update_state to convert raw logits to predicted class labels
# before passing to the standard MeanIoU metric.
# =============================================================================
class MyMeanIoU(keras.metrics.MeanIoU):
    def update_state(self, y_true, y_pred, sample_weight=None):
        # Convert per-pixel class logits to predicted class index
        pred_mask = tf.argmax(y_pred, axis=-1)[..., tf.newaxis]
        return super().update_state(y_true, pred_mask, sample_weight)

model.compile(
    optimizer='adam',
    # SparseCategoricalCrossentropy expects integer class labels (not one-hot)
    # from_logits=True because our output layer has no softmax activation
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=[
        'accuracy',
        MyMeanIoU(num_classes=4, name='mean_iou'),
    ]
)

# =============================================================================
# TRAINING CALLBACKS
# - ModelCheckpoint: saves the best model weights based on validation Mean IoU
# - EarlyStopping: halts training if validation IoU stops improving
# - ReduceLROnPlateau: reduces learning rate when validation loss plateaus,
#   helping the model escape local minima and continue improving
# =============================================================================
callbacks = [
    keras.callbacks.ModelCheckpoint(
        str(ROOT / 'unet_4class_best.keras'),
        save_best_only=True, monitor='val_mean_iou', mode='max'),
    keras.callbacks.EarlyStopping(
        patience=10, min_delta=0.005,
        monitor='val_mean_iou', mode='max'),
    keras.callbacks.ReduceLROnPlateau(
        patience=5, factor=0.5, monitor='val_loss', verbose=1),
]

# Train the model
history = model.fit(
    ds_train,
    epochs=EPOCHS,
    validation_data=ds_val,
    callbacks=callbacks,
)

# Evaluate on held-out test set using best saved weights
print('\nTest set evaluation:')
model.load_weights(str(ROOT / 'unet_4class_best.keras'))
model.evaluate(ds_test)

# =============================================================================
# VISUALIZATION
# =============================================================================

def create_mask(pred):
    """Convert per-pixel class logits to a predicted segmentation mask."""
    return tf.argmax(pred, axis=-1)[..., tf.newaxis]

def show_prediction(ds, n=3):
    """
    Display n sample predictions side by side with their input CT image
    and ground truth mask. Saves output to a PNG file.
    Color scale: 0=background, 1=hemorrhage, 2=brain tissue, 3=skull
    """
    fig, axes = plt.subplots(n, 3, figsize=(12, n * 4))
    for i, (img, mask) in enumerate(ds.unbatch().take(n)):
        pred = model.predict(img[tf.newaxis, ...], verbose=0)
        pred_mask = create_mask(pred)[0]
        axes[i, 0].imshow(img.numpy())
        axes[i, 0].set_title('Input CT')
        axes[i, 1].imshow(mask.numpy()[..., 0], cmap='viridis', vmin=0, vmax=3)
        axes[i, 1].set_title('True Mask')
        axes[i, 2].imshow(pred_mask.numpy()[..., 0], cmap='viridis', vmin=0, vmax=3)
        axes[i, 2].set_title('Predicted Mask')
        for ax in axes[i]:
            ax.axis('off')
    plt.tight_layout()
    plt.savefig(str(ROOT / 'Unet 4Class prediction.png'), dpi=150)
    plt.show()

show_prediction(ds_test, n=3)

# Plot training history: loss, accuracy, and Mean IoU across epochs
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(15, 4))
h = history.history
ax1.plot(h['loss'], label='Train');     ax1.plot(h['val_loss'], label='Val')
ax1.set_title('Loss'); ax1.legend()
ax2.plot(h['accuracy'], label='Train'); ax2.plot(h['val_accuracy'], label='Val')
ax2.set_title('Accuracy'); ax2.legend()
ax3.plot(h['mean_iou'], label='Train'); ax3.plot(h['val_mean_iou'], label='Val')
ax3.set_title('Mean IoU'); ax3.legend()
plt.tight_layout()
plt.savefig(str(ROOT / 'Unet 4class history.png'), dpi=150)
plt.show()
